# jefscad playground

Interactive scratch pad for exploring the `jefscad` Python package.

**Prerequisites** — from the repo root:
```bash
export VIRTUAL_ENV=$(pwd)/.venv
.venv/bin/maturin develop --features extension-module
.venv/bin/jupyter lab
```

**After editing Python code** (`python/jefscad/__init__.py` etc.): re-run any cell — autoreload picks up the change automatically.

**After editing Rust code** (`jefscad/src/`): run `maturin develop` in a terminal, then use **Kernel → Restart Kernel** in JupyterLab and re-run from the top.

In [1]:
# Enable autoreload so Python-side changes are picked up without a kernel restart.
# For Rust-side changes you still need to run `maturin develop` then restart the kernel.
%load_ext autoreload
%autoreload 2

In [2]:
import math

In [3]:
import jefscad
print("jefscad loaded successfully")
print("Available names:", [n for n in dir(jefscad) if not n.startswith('_')])

jefscad loaded successfully
Available names: ['Node', 'cone', 'cuboid', 'cylinder', 'difference', 'intersection', 'select_closest_to', 'select_contains', 'select_largest', 'sphere', 'union']


## The jefscad `help` section

This give an overview of the public API for the package

In [4]:
help(jefscad)

Help on package jefscad:

NAME
    jefscad - jefscad — solid modeling language.

DESCRIPTION
    The core implementation is in Rust (via pyo3). This module re-exports
    everything from the compiled Rust extension (jefscad._jefscad) and is
    the place to add pure-Python helpers, submodules, or documentation
    augmentations on top of the Rust layer.

PACKAGE CONTENTS
    _jefscad

CLASSES
    builtins.object
        builtins.Node
    
    class Node(object)
     |  A CSG node: a primitive or operation with an associated transform stack.
     |  
     |  Create nodes with the module-level constructors (`sphere`, `cuboid`, etc.)
     |  and chain transforms with the methods below. All transform methods return
     |  a **new** Node; the original is never mutated.
     |  
     |  Methods defined here:
     |  
     |  __repr__(self, /)
     |      Return repr(self).
     |  
     |  __str__(self, /)
     |      Return str(self).
     |  
     |  rot_aa(self, /, axis, angle_rad)
     

## Scratch space

Add cells below to explore new features as the package grows.

In [13]:
# Create a primative with a few transforms
my_cone = jefscad.cone(r=0.5, h=1).translate(0,0,-0.5).rot_x(math.pi/2)
# The __repr__ or debug style print with relevant info
my_cone

Node { base: Prim(Cone { r: 0.5, h: 1.0 }), transforms: [Translation { delta: [0.0, 0.0, -0.5] }, RotationAA { axis: [1.0, 0.0, 0.0], angle: 1.5707963267948966 }] }

In [14]:
# The more human readable tree strucure of the CSG primative
print(my_cone)

cone(r=0.5, h=1.0)
└── transforms
    ├── translate(dx=0.0, dy=0.0, dz=-0.5)
    └── rot_aa(ax=1.0, ay=0.0, az=0.0, angle=1.5708)



In [15]:
# Create a second primative that with a different order of tranforms that are equivalent
cone_2 = jefscad.cone(r=0.5, h=1).rot_x(math.pi/2).translate(0,-0.5,0)
# the human readable structure
print(cone_2)

cone(r=0.5, h=1.0)
└── transforms
    ├── rot_aa(ax=1.0, ay=0.0, az=0.0, angle=1.5708)
    └── translate(dx=0.0, dy=-0.5, dz=0.0)



In [18]:
# The geom_id should allow use to compare geometries
my_cone.geom_id == cone_2.geom_id

True

In [20]:
# the prov_id is unique for each
my_cone.prov_id==cone_2.prov_id, my_cone.prov_id, cone_2.prov_id
# the prov_id is a simple counter global atomic counter, so these should be small numbers

(False, 9, 12)

In [23]:
# lets create a sphere with two conical holes
holy_sphere = jefscad.difference(
    # start with a sphere
    jefscad.sphere(r=0.5),
    # subtract out the first cone
    my_cone,
    # flip the other cone, and subtract that as well
    cone_2.rot_x(math.pi)
)
# print the csg_tree at this state
print(holy_sphere)

difference
├── base
│   └── sphere(r=0.5)
└── subtract
    ├── cone(r=0.5, h=1.0)
    │   └── transforms
    │       ├── translate(dx=0.0, dy=0.0, dz=-0.5)
    │       └── rot_aa(ax=1.0, ay=0.0, az=0.0, angle=1.5708)
    └── cone(r=0.5, h=1.0)
        └── transforms
            ├── rot_aa(ax=1.0, ay=0.0, az=0.0, angle=1.5708)
            ├── translate(dx=0.0, dy=-0.5, dz=0.0)
            └── rot_aa(ax=1.0, ay=0.0, az=0.0, angle=3.1416)

